# Vanilla CMUNeXt Training

This notebook trains the vanilla CMUNeXt baseline for retinal thin-vessel segmentation in Google Colab.

Expected dataset layout:

```text
DATA_ROOT/
  images/
    sample_001.png
  masks/
    sample_001.png
```

Image and mask filenames must share the same stem.

In [ ]:
!pip install -q torch torchvision tqdm pillow pandas kaggle

import os
import sys
from pathlib import Path
from google.colab import userdata
from kaggle.api.kaggle_api_extended import KaggleApi

# Upload this project folder to the Colab session, clone it, or keep it under /content.
PROJECT_ROOT = '/content/Research_Formal'

# Store KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets before running.
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')
if not kaggle_username or not kaggle_key:
    raise RuntimeError('Add KAGGLE_USERNAME and KAGGLE_KEY to Colab Secrets before running this notebook.')
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key

# Kaggle dataset: https://www.kaggle.com/datasets/ipythonx/retinal-vessel-segmentation
KAGGLE_DATASET = 'ipythonx/retinal-vessel-segmentation'
DATASET_SUBDIR = ''  # Example: 'DRIVE' if the unzipped dataset has a DRIVE/ folder.

dataset_dir = Path('/content/datasets') / KAGGLE_DATASET.split('/')[-1]
dataset_dir.mkdir(parents=True, exist_ok=True)

api = KaggleApi()
api.authenticate()
if not any(dataset_dir.iterdir()):
    api.dataset_download_files(KAGGLE_DATASET, path=dataset_dir, unzip=True, quiet=False)

DATA_ROOT = dataset_dir / DATASET_SUBDIR if DATASET_SUBDIR else dataset_dir
IMAGE_DIR = DATA_ROOT / 'images'
MASK_DIR = DATA_ROOT / 'masks'

project_root = Path(PROJECT_ROOT)
sys.path.insert(0, str(project_root / 'src'))

OUTPUT_DIR = project_root / 'outputs' / 'vanilla_cmunext'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project:', project_root)
print('Dataset root:', DATA_ROOT)
print('Images:', IMAGE_DIR)
print('Masks:', MASK_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from research_formal.models import cmunext_l
from research_formal.training import (
    BCEDiceLoss,
    VesselSegmentationDataset,
    binary_segmentation_metrics,
    count_trainable_parameters,
    split_image_mask_pairs,
)

SEED = 41
IMAGE_SIZE = 512
BATCH_SIZE = 4
EPOCHS = 50
LR = 1e-3
VAL_FRACTION = 0.2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
train_pairs, val_pairs = split_image_mask_pairs(
    IMAGE_DIR,
    MASK_DIR,
    val_fraction=VAL_FRACTION,
    seed=SEED,
)

train_ds = VesselSegmentationDataset(train_pairs, image_size=IMAGE_SIZE, augment=True)
val_ds = VesselSegmentationDataset(val_pairs, image_size=IMAGE_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

print('Train images:', len(train_ds))
print('Val images:', len(val_ds))

In [ ]:
# Official CMUNeXt-L configuration from FengheTan9/CMUNeXt:
# dims=[32, 64, 128, 256, 512], depths=[1, 1, 1, 6, 3], kernels=[3, 3, 7, 7, 7]
model = cmunext_l(input_channel=3, num_classes=1).to(device)
criterion = BCEDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print('Trainable parameters:', count_trainable_parameters(model))

In [ ]:
def run_validation():
    model.eval()
    loss_total = 0.0
    metric_totals = {'dice': 0.0, 'iou': 0.0, 'sensitivity': 0.0, 'specificity': 0.0, 'accuracy': 0.0}
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            logits = model(images)
            loss_total += criterion(logits, masks).item()
            metrics = binary_segmentation_metrics(logits, masks)
            for key in metric_totals:
                metric_totals[key] += metrics[key]
    count = max(1, len(val_loader))
    return loss_total / count, {key: value / count for key, value in metric_totals.items()}

best_dice = -1.0
history = []
history_csv = OUTPUT_DIR / 'training_history.csv'

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for images, masks in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}'):
        images = images.to(device)
        masks = masks.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    val_loss, val_metrics = run_validation()
    train_loss /= max(1, len(train_loader))
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, **val_metrics}
    history.append(row)
    pd.DataFrame(history).to_csv(history_csv, index=False)
    print(row)

    if val_metrics['dice'] > best_dice:
        best_dice = val_metrics['dice']
        torch.save({'model_state_dict': model.state_dict(), 'config': {'image_size': IMAGE_SIZE, 'variant': 'CMUNeXt-L'}}, OUTPUT_DIR / 'best_vanilla_cmunext_l.pt')
        print('Saved best checkpoint')

In [ ]:
history_df = pd.read_csv(OUTPUT_DIR / 'training_history.csv')
history_df.tail()